In [60]:
import neo4j

import csv

import math
import numpy as np
import pandas as pd

import psycopg2
import gmaps
import gmaps.geojson_geometries

from geographiclib.geodesic import Geodesic

In [61]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

In [62]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [63]:
cursor = connection.cursor()

In [64]:
def my_read_csv_file(file_name, limit):
    "read the csv file and print only the first limit rows"
    
    csv_file = open(file_name, "r")
    
    csv_data = csv.reader(csv_file)
    
    i = 0
    
    for row in csv_data:
        i += 1
        if i <= limit:
            print(row)
            
    print("\nPrinted ", min(limit, i), "lines of ", i, "total lines.")

In [65]:
def my_recursive_print_json(j, level = -1):
    "given a json object print it"
    
    level += 1
    
    spaces = "    "
    
    if type(j) is dict:
        dict_2_list = list(j.keys())
        for k in dict_2_list:
            print(spaces * level + k)
            my_recursive_print_json(j[k], level)
            
    elif type(j) is list:
        for (i, l) in enumerate(j):
            print(spaces * level + "[" + str(i) + "]")
            my_recursive_print_json(l, level)
                  
    else:
        print(spaces * level + "value:", str(j))
                  

In [66]:
def my_read_nested_json(file_name):
    "given a file of json, read it and parse it meaningfully"
    
    f = open(file_name, "r")
    
    j = json.load(f)
    
    f.close
    
    my_recursive_print_json(j)

# Read CSV files for stations, real time estimation and routes

In [67]:
my_read_csv_file('etd.csv', 10)

['Station Name', 'Destination', 'Minutes', 'Platform', 'Direction', 'Length', 'Color', 'Hexcolor', 'Bikeflag', 'Delay', 'Cancelflag', 'Dynamicflag']
['Lake Merritt', 'Berryessa', '2', '1', 'South', '8', 'ORANGE', '#ff9933', '1', '89', '0', '0']
['Lake Merritt', 'Berryessa', '19', '1', 'North', '10', 'GREEN', '#339933', '1', '236', '0', '0']
['Lake Merritt', 'Berryessa', '31', '1', 'South', '8', 'ORANGE', '#ff9933', '1', '0', '0', '0']
['Lake Merritt', 'Daly City', '9', '2', 'South', '10', 'GREEN', '#339933', '1', '140', '0', '0']
['Lake Merritt', 'Daly City', '23', '2', 'South', '8', 'BLUE', '#0099cc', '1', '0', '0', '0']
['Lake Merritt', 'Daly City', '37', '2', 'South', '10', 'GREEN', '#339933', '1', '0', '0', '0']
['Lake Merritt', 'Dublin/Pleasanton', 'Leaving', '1', 'South', '8', 'BLUE', '#0099cc', '1', '198', '0', '0']
['Lake Merritt', 'Dublin/Pleasanton', '28', '1', 'South', '8', 'BLUE', '#0099cc', '1', '0', '0', '0']
['Lake Merritt', 'Dublin/Pleasanton', '58', '1', 'South', '10',

In [68]:
my_read_csv_file('route_info.csv', 10)

['name', 'abbr', 'routeID', 'number', 'origin', 'destination', 'direction', 'hexcolor', 'color', 'num_stns', 'station']
['Coliseum to Oakland Airport', 'COLS-OAKL', 'ROUTE 20', '20', 'COLS', 'OAKL', 'South', '#D5CFA3', 'BEIGE', '2', "['COLS', 'OAKL']"]
['Dublin/Pleasanton to Daly City', 'DUBL-DALY', 'ROUTE 11', '11', 'DUBL', 'DALY', 'South', '#AA5599', 'BLUE', '18', "['DUBL', 'WDUB', 'CAST', 'BAYF', 'SANL', 'COLS', 'FTVL', 'LAKE', 'WOAK', 'EMBR', 'MONT', 'POWL', 'CIVC', '16TH', '24TH', 'GLEN', 'BALB', 'DALY']"]
['Daly City to Dublin/Pleasanton', 'DALY-DUBL', 'ROUTE 12', '12', 'DALY', 'DUBL', 'North', '#0099CC', 'BLUE', '18', "['DALY', 'BALB', 'GLEN', '24TH', '16TH', 'CIVC', 'POWL', 'MONT', 'EMBR', 'WOAK', 'LAKE', 'FTVL', 'COLS', 'SANL', 'BAYF', 'CAST', 'WDUB', 'DUBL']"]
['Berryessa/North San Jose to Daly City', 'BERY-DALY', 'ROUTE 5', '5', 'BERY', 'DALY', 'South', '#AA5599', 'GREEN', '22', "['BERY', 'MLPT', 'WARM', 'FRMT', 'UCTY', 'SHAY', 'HAYW', 'BAYF', 'SANL', 'COLS', 'FTVL', 'LAKE',

In [69]:
my_read_csv_file('station_info.csv', 10)

['name', 'abbr', 'gtfs_latitude', 'gtfs_longitude', 'address', 'city', 'county', 'state', 'zipcode', 'north_routes', 'south_routes']
['12th St. Oakland City Center', '12TH', '37.803768', '-122.271450', '1245 Broadway', 'Oakland', 'alameda', 'CA', '94612', "['ROUTE 2', 'ROUTE 3', 'ROUTE 8']", "['ROUTE 1', 'ROUTE 4', 'ROUTE 7']"]
['16th St. Mission', '16TH', '37.765062', '-122.419694', '2000 Mission Street', 'San Francisco', 'sanfrancisco', 'CA', '94110', "['ROUTE 2', 'ROUTE 6', 'ROUTE 8', 'ROUTE 12']", "['ROUTE 1', 'ROUTE 5', 'ROUTE 7', 'ROUTE 11']"]
['19th St. Oakland', '19TH', '37.808350', '-122.268602', '1900 Broadway', 'Oakland', 'alameda', 'CA', '94612', "['ROUTE 2', 'ROUTE 3', 'ROUTE 8']", "['ROUTE 1', 'ROUTE 4', 'ROUTE 7']"]
['24th St. Mission', '24TH', '37.752470', '-122.418143', '2800 Mission Street', 'San Francisco', 'sanfrancisco', 'CA', '94110', "['ROUTE 2', 'ROUTE 6', 'ROUTE 8', 'ROUTE 12']", "['ROUTE 1', 'ROUTE 5', 'ROUTE 7', 'ROUTE 11']"]
['Antioch', 'ANTC', '37.995388', 

# Drop and Create Tables

In [70]:
connection.rollback()

query = """

drop table if exists real_time

"""

cursor.execute(query)

connection.commit()

In [71]:
connection.rollback()

query = """

drop table if exists stations

"""

cursor.execute(query)

connection.commit()

In [72]:
connection.rollback()

query = """

drop table if exists route

"""

cursor.execute(query)

connection.commit()

In [73]:
connection.rollback()

query = """

create table real_time (
    station_name varchar(64),
    destination varchar(32),
    minutes varchar(32),
    platform varchar(32),
    direction varchar(32),
    length varchar(32),
    color varchar(32),
    hexcolor varchar(32),
    bikeflag numeric(3),
    delay numeric(5),
    cancelflag numeric(3), 
    dynamicflag numeric(3))

"""

cursor.execute(query)

connection.commit()

In [74]:
connection.rollback()

query = """

create table stations (
 station varchar(32),
 latitude numeric(9,6),
 longitude numeric(9,6),
 transfer_time numeric(3),
 primary key (station))
"""

cursor.execute(query)

connection.commit()

In [75]:
connection.rollback()

query = """

create table route (
    name varchar(64),
    abbr varchar(32),
    routeID varchar(32),
    number varchar(32),
    origin varchar(32),
    destination varchar(32),
    direction varchar(32),
    hexcolor varchar(32),
    color varchar(32),
    num_stns numeric(3),
    station varchar(1000))

"""

cursor.execute(query)

connection.commit()

In [76]:
connection.rollback()

query = """

copy real_time
from '/user/projects/project-3-AnaZapataG/exercise/etd.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

In [77]:
connection.rollback()

query = """

copy route
from '/user/projects/project-3-AnaZapataG/exercise/route_info.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

In [78]:
connection.rollback()

query = """

copy stations
from '/user/projects/project-3-AnaZapataG/exercise/stations.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

In [79]:
my_read_csv_file("stations.csv", limit=10)

['station', 'latitude', 'longitude', 'transfer_time']
['12th Street', '37.803608', '-122.272006', '282']
['16th Street Mission', '37.764847', '-122.420042', '287']
['19th Street', '37.807869', '-122.26898', '67']
['24th Street Mission', '37.752', '-122.4187', '277']
['Antioch', '37.996281', '-121.783404', '0']
['Ashby', '37.853068', '-122.269957', '299']
['Balboa Park', '37.721667', '-122.4475', '48']
['Bay Fair', '37.697', '-122.1265', '63']
['Berryessa', '37.368361', '-121.874655', '288']

Printed  10 lines of  51 total lines.


# All real time estimations

In [80]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from real_time
order by station_name

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station_name,destination,minutes,platform,direction,length,color,hexcolor,bikeflag,delay,cancelflag,dynamicflag
0,12th St. Oakland City Center,SF Airport,26,2,South,10,YELLOW,#ffff33,1,0,0,0
1,12th St. Oakland City Center,Antioch,23,3,North,8,YELLOW,#ffff33,1,0,0,0
2,12th St. Oakland City Center,Antioch,53,3,North,10,YELLOW,#ffff33,1,0,0,0
3,12th St. Oakland City Center,Antioch,83,3,North,10,YELLOW,#ffff33,1,0,0,0
4,12th St. Oakland City Center,Berryessa,Leaving,2,South,8,ORANGE,#ff9933,1,145,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
413,West Oakland,Daly City,14,1,South,10,GREEN,#339933,1,115,0,0
414,West Oakland,Berryessa,39,2,North,10,GREEN,#339933,1,0,0,0
415,West Oakland,Antioch,48,2,North,10,YELLOW,#ffff33,1,0,0,0
416,West Oakland,Antioch,78,2,North,10,YELLOW,#ffff33,1,0,0,0


# All routes

In [81]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from route
order by name

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,name,abbr,routeid,number,origin,destination,direction,hexcolor,color,num_stns,station
0,Antioch to SFIA/Millbrae,ANTC-MLBR,ROUTE 1,1,ANTC,MLBR,South,#FFFF33,YELLOW,28,"['ANTC', 'PCTR', 'PITT', 'NCON', 'CONC', 'PHIL..."
1,Berryessa/North San Jose to Daly City,BERY-DALY,ROUTE 5,5,BERY,DALY,South,#AA5599,GREEN,22,"['BERY', 'MLPT', 'WARM', 'FRMT', 'UCTY', 'SHAY..."
2,Berryessa/North San Jose to Richmond,BERY-RICH,ROUTE 3,3,BERY,RICH,North,#FF0000,ORANGE,21,"['BERY', 'MLPT', 'WARM', 'FRMT', 'UCTY', 'SHAY..."
3,Coliseum to Oakland Airport,COLS-OAKL,ROUTE 20,20,COLS,OAKL,South,#D5CFA3,BEIGE,2,"['COLS', 'OAKL']"
4,Daly City to Berryessa/North San Jose,DALY-BERY,ROUTE 6,6,DALY,BERY,North,#FF9933,GREEN,22,"['DALY', 'BALB', 'GLEN', '24TH', '16TH', 'CIVC..."
5,Daly City to Dublin/Pleasanton,DALY-DUBL,ROUTE 12,12,DALY,DUBL,North,#0099CC,BLUE,18,"['DALY', 'BALB', 'GLEN', '24TH', '16TH', 'CIVC..."
6,Dublin/Pleasanton to Daly City,DUBL-DALY,ROUTE 11,11,DUBL,DALY,South,#AA5599,BLUE,18,"['DUBL', 'WDUB', 'CAST', 'BAYF', 'SANL', 'COLS..."
7,Millbrae/Daly City to Richmond,SFIA-RICH,ROUTE 8,8,SFIA,RICH,North,#FF0000,RED,24,"['SFIA', 'MLBR', 'SBRN', 'SSAN', 'COLM', 'DALY..."
8,Millbrae/SFIA to Antioch,MLBR-ANTC,ROUTE 2,2,MLBR,ANTC,North,#FFFF33,YELLOW,28,"['MLBR', 'SFIA', 'SBRN', 'SSAN', 'COLM', 'DALY..."
9,Richmond to Berryessa/North San Jose,RICH-BERY,ROUTE 4,4,RICH,BERY,South,#FF9933,ORANGE,21,"['RICH', 'DELN', 'PLZA', 'NBRK', 'DBRK', 'ASHB..."


# All stations

In [82]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from stations


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station,latitude,longitude,transfer_time
0,12th Street,37.803608,-122.272006,282
1,16th Street Mission,37.764847,-122.420042,287
2,19th Street,37.807869,-122.268980,67
3,24th Street Mission,37.752000,-122.418700,277
4,Antioch,37.996281,-121.783404,0
5,Ashby,37.853068,-122.269957,299
6,Balboa Park,37.721667,-122.447500,48
7,Bay Fair,37.697000,-122.126500,63
8,Berryessa,37.368361,-121.874655,288
9,Castro Valley,37.690748,-122.075679,0


# Neo4j set up

In [83]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [84]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [85]:
def my_neo4j_number_nodes_relationships():
    "print the number of nodes and relationships"
   
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    print("-------------------------")
    print("  Nodes:", number_nodes)
    print("  Relationships:", number_relationships)
    print("-------------------------")


In [86]:
def my_neo4j_create_node(station_name):
    "create a node with label Station"
    
    query = """
    
    CREATE (:Station {name: $station_name})
    
    """
    
    session.run(query, station_name=station_name)
    

In [87]:
def my_neo4j_create_relationship_one_way(from_station, to_station, weight):
    "create a relationship one way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)

In [88]:
def my_neo4j_create_relationship_two_way(from_station, to_station, weight):
    "create relationships two way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to),
           (to)-[:LINK {weight: $weight}]->(from)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)

# Shortest path

In [89]:
def my_neo4j_shortest_path(from_station, to_station):
    "given a from station and to station, run and print the shortest path"
    
    query = "CALL gds.graph.drop('ds_graph', false)"
    session.run(query)

    query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
    session.run(query)

    query = """

    MATCH (source:Station {name: $source}), (target:Station {name: $target})
    CALL gds.shortestPath.dijkstra.stream(
        'ds_graph', 
        { sourceNode: source, 
          targetNode: target, 
          relationshipWeightProperty: 'weight'
        }
    )
    YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
    RETURN
        gds.util.asNode(sourceNode).name AS from,
        gds.util.asNode(targetNode).name AS to,
        totalCost,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodes,
        costs
    ORDER BY index

    """

    result = session.run(query, source=from_station, target=to_station)
    
    for r in result:
        
        total_cost = int(r['totalCost'])
        
        print("\n--------------------------------")
        print("   Total Cost: ", total_cost)
        print("   Minutes: ", round(total_cost / 60.0,1))
        print("--------------------------------")
        
        nodes = r['nodes']
        costs = r['costs']
        
        i = 0
        previous = 0
        
        for n in nodes:
            
            print(n + ", " + str(int(costs[i]) - previous)  + ", " + str(int(costs[i])))
            
            previous = int(costs[i])
            i += 1

In [90]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [91]:
session = driver.session(database="neo4j")

In [92]:
my_neo4j_wipe_out_database()

In [93]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 0
  Relationships: 0
-------------------------


In [94]:
connection.rollback()

query = """

select station
from stations
order by station

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    
    my_neo4j_create_node('depart ' + station)
    my_neo4j_create_node('arrive ' + station)

In [95]:
connection.rollback()

query = """

select station, line
from lines
order by station, line

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    line = row[1]
    
    depart = 'depart ' + station
    arrive = 'arrive ' + station
    line_station = line + ' ' + station
    
    my_neo4j_create_node(line_station)
    my_neo4j_create_relationship_one_way(depart, line_station, 0)
    my_neo4j_create_relationship_one_way(line_station, arrive, 0)

In [96]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 214
  Relationships: 228
-------------------------


In [97]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 214
  Relationships: 228
-------------------------


In [98]:
connection.rollback()

query = """

select a.station, a.line as from_line, b.line as to_line, s.transfer_time
from lines a
     join lines b
       on a.station = b.station and a.line <> b.line 
     join stations s
       on a.station = s.station
order by 1, 2, 3

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    from_line = row[1]
    to_line = row[2]
    transfer_time = int(row[3])
    
    from_station = from_line + ' ' + station
    to_station = to_line + ' ' + station
    
    my_neo4j_create_relationship_one_way(from_station, to_station, transfer_time)
    

In [99]:
connection.rollback()

query = """

select a.line, a.station as from_station, b.station as to_station, t.travel_time
from lines a
  join lines b
    on a.line = b.line and b.sequence = (a.sequence + 1)
  join travel_times t
    on (a.station = t.station_1 and b.station = t.station_2)
        or (a.station = t.station_2 and b.station = t.station_1)
order by line, from_station, to_station

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    line = row[0]
    from_station = line + ' ' + row[1]
    to_station = line + ' ' + row[2]
    travel_time = int(row[3])
    
    my_neo4j_create_relationship_two_way(from_station, to_station, travel_time)

In [100]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 214
  Relationships: 652
-------------------------


In [101]:
my_neo4j_shortest_path('depart Downtown Berkeley', 'arrive Ashby')


--------------------------------
   Total Cost:  180
   Minutes:  3.0
--------------------------------
depart Downtown Berkeley, 0, 0
red Downtown Berkeley, 0, 0
red Ashby, 180, 180
arrive Ashby, 0, 180


In [102]:
my_neo4j_shortest_path('depart Downtown Berkeley', 'arrive North Berkeley')


--------------------------------
   Total Cost:  120
   Minutes:  2.0
--------------------------------
depart Downtown Berkeley, 0, 0
red Downtown Berkeley, 0, 0
red North Berkeley, 120, 120
arrive North Berkeley, 0, 120


In [103]:
my_neo4j_shortest_path('depart Downtown Berkeley', 'arrive El Cerrito Plaza')


--------------------------------
   Total Cost:  300
   Minutes:  5.0
--------------------------------
depart Downtown Berkeley, 0, 0
orange Downtown Berkeley, 0, 0
orange North Berkeley, 120, 120
orange El Cerrito Plaza, 180, 300
arrive El Cerrito Plaza, 0, 300


In [104]:
my_neo4j_shortest_path('depart Downtown Berkeley', 'arrive Rockridge')


--------------------------------
   Total Cost:  719
   Minutes:  12.0
--------------------------------
depart Downtown Berkeley, 0, 0
orange Downtown Berkeley, 0, 0
orange Ashby, 180, 180
orange MacArthur, 240, 420
yellow MacArthur, 59, 479
yellow Rockridge, 240, 719
arrive Rockridge, 0, 719


In [105]:
my_neo4j_shortest_path('depart Downtown Berkeley', 'arrive Lake Merritt')


--------------------------------
   Total Cost:  900
   Minutes:  15.0
--------------------------------
depart Downtown Berkeley, 0, 0
orange Downtown Berkeley, 0, 0
orange Ashby, 180, 180
orange MacArthur, 240, 420
orange 19th Street, 180, 600
orange 12th Street, 120, 720
orange Lake Merritt, 180, 900
arrive Lake Merritt, 0, 900


# Query for real time estimation from BART API

In [106]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from real_time where station_name = 'Downtown Berkeley' and color = 'ORANGE'

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station_name,destination,minutes,platform,direction,length,color,hexcolor,bikeflag,delay,cancelflag,dynamicflag
0,Downtown Berkeley,Berryessa,15,2,South,8,ORANGE,#ff9933,1,0,0,0
1,Downtown Berkeley,Berryessa,45,2,South,8,ORANGE,#ff9933,1,0,0,0
2,Downtown Berkeley,Richmond,7,1,North,8,ORANGE,#ff9933,1,148,0,0


In [30]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from real_time where station_name = 'Downtown Berkeley' and color = 'RED'

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station_name,destination,minutes,platform,direction,length,color,hexcolor,bikeflag,delay,cancelflag,dynamicflag
0,Downtown Berkeley,Millbrae/SFO,33,2,South,10,RED,#ff0000,1,316,0,0
1,Downtown Berkeley,Millbrae/SFO,58,2,South,10,RED,#ff0000,1,0,0,0
2,Downtown Berkeley,Richmond,4,1,North,10,RED,#ff0000,1,1026,0,0
3,Downtown Berkeley,Richmond,17,1,North,10,RED,#ff0000,1,0,0,0


# Map for all of the stations

In [38]:
f = open('gmap_api_key.txt', 'r')
my_api_key = f.read()
f.close()

gmaps.configure(api_key=my_api_key)

In [32]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from real_time where station_name = 'Downtown Berkeley' and color = 'RED' and minutes ='Leaving'

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station_name,destination,minutes,platform,direction,length,color,hexcolor,bikeflag,delay,cancelflag,dynamicflag


In [108]:
sather_gate_berkeley = (37.870260430419115, -122.25950168579497)

gmaps.figure(center=sather_gate_berkeley, zoom_level=9)

Figure(layout=FigureLayout(height='420px'))

In [109]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=9, map_type='TERRAIN')

Figure(layout=FigureLayout(height='420px'))

In [110]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=9, map_type='SATELLITE')

Figure(layout=FigureLayout(height='420px'))

In [111]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=9, map_type='HYBRID')

Figure(layout=FigureLayout(height='420px'))

In [112]:
figure_layout = {
    'width': '300px',
    'height': '500px',
    'border': '1px solid black',
    'padding': '1px'
}

gmaps.figure(center=sather_gate_berkeley, zoom_level=9, layout=figure_layout)

Figure(layout=FigureLayout(border='1px solid black', height='500px', padding='1px', width='300px'))

In [113]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=12)

Figure(layout=FigureLayout(height='420px'))

In [114]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=15)

Figure(layout=FigureLayout(height='420px'))

In [115]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=17)

Figure(layout=FigureLayout(height='420px'))

In [116]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=19)

Figure(layout=FigureLayout(height='420px'))

In [117]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=21)

Figure(layout=FigureLayout(height='420px'))

In [118]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select station,
    latitude,
    longitude
from stations
order by station

"""
df = my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)
df

,station,latitude,longitude
0,12th Street,37.803608,-122.272006
1,16th Street Mission,37.764847,-122.420042
2,19th Street,37.807869,-122.268980
3,24th Street Mission,37.752000,-122.418700
4,Antioch,37.996281,-121.783404
5,Ashby,37.853068,-122.269957
6,Balboa Park,37.721667,-122.447500
7,Bay Fair,37.697000,-122.126500
8,Berryessa,37.368361,-121.874655
9,Castro Valley,37.690748,-122.075679


In [119]:
fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=8)

df_markers = df[['latitude','longitude']]

marker_layer = gmaps.marker_layer(df_markers)

fig.add_layer(marker_layer)

fig

Figure(layout=FigureLayout(height='420px'))